# Surface Code Noise — Coherent vs Pauli

> 🚀 **Try Maestro GPU mode with a free trial.**
> Sign up at **[maestro.qoroquantum.net](https://maestro.qoroquantum.net)** — no credit card required.

## Why This Matters

Real quantum hardware has both **stochastic (Pauli)** and **coherent** noise. Stabiliser simulators like Stim can only model Pauli noise. Maestro's MPS backend captures **both** — revealing error structures invisible to Pauli-only models.

This notebook simulates noisy CX networks in surface code topology and compares:
- **Pauli noise** — random rotations (incoherent, partially cancel)
- **Coherent noise** — systematic over-rotations (accumulate constructively)
- **Multiple backends** — Stabilizer, PauliPropagator, MPS all in one tool

## Setup

```bash
pip install qoro-maestro matplotlib numpy
```

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

import maestro
from model import SurfaceCodeModel

# Configuration
NOISE_STRENGTHS = [0.005, 0.01, 0.015, 0.02, 0.03, 0.04, 0.05]
N_ROUNDS = 3       # CX rounds per circuit
N_SAMPLES = 20     # Pauli noise averaging
CHI = 32           # MPS bond dimension

USE_GPU = False    # Set True for GPU mode
SIM_TYPE = maestro.SimulatorType.Gpu if USE_GPU else maestro.SimulatorType.QCSim


def build_data_only_observables(n_data):
    obs = []
    for i in range(n_data):
        pauli = ['I'] * n_data
        pauli[i] = 'Z'
        obs.append(''.join(pauli))
    return obs


def build_logical_z(model):
    pauli = ['I'] * model.n_data
    for q in model.logical_z_qubits:
        pauli[q] = 'Z'
    return ''.join(pauli)


def measure_noise_impact(model, noise_type, p, n_samples=1):
    """Measure noise impact on data qubits via CX noise network."""
    data_obs = build_data_only_observables(model.n_data)
    logical_z_obs = build_logical_z(model)
    all_obs = data_obs + [logical_z_obs]

    all_data_z, all_logical_z = [], []
    for sample in range(n_samples):
        seed = sample if noise_type == 'pauli' else None
        qc = model.build_noisy_cx_network(
            n_rounds=N_ROUNDS, noise_type=noise_type,
            noise_strength=p, seed=seed)
        result = qc.estimate(all_obs, maestro.SimulatorConfig(
            simulator_type=SIM_TYPE,
            simulation_type=maestro.SimulationType.MatrixProductState,
            max_bond_dimension=CHI))
        exp_vals = result['expectation_values']
        all_data_z.append(np.array(exp_vals[:model.n_data]))
        all_logical_z.append(exp_vals[-1])

    return {
        'data_z': np.mean(all_data_z, axis=0),
        'logical_z': float(np.mean(all_logical_z)),
    }

print('Ready.')

---

## 1. Noise Comparison — Coherent vs Pauli

Sweep noise strength and compare logical Z fidelity under Pauli and coherent noise at d=3 and d=5.

In [ ]:
all_results = []

for d in [3, 5]:
    model = SurfaceCodeModel(d)
    print(f'\nd={d}: {model.n_data} data qubits, {model.n_total} total')
    
    pauli_z, coherent_z = [], []
    
    for p in NOISE_STRENGTHS:
        t0 = time.time()
        pauli = measure_noise_impact(model, 'pauli', p, n_samples=N_SAMPLES)
        coherent = measure_noise_impact(model, 'coherent', p, n_samples=1)
        elapsed = time.time() - t0
        pauli_z.append(pauli['logical_z'])
        coherent_z.append(coherent['logical_z'])
        print(f'  ε={p:.3f}  Pauli={pauli["logical_z"]:+.4f}  '
              f'Coherent={coherent["logical_z"]:+.4f}  ({elapsed:.1f}s)')
    
    all_results.append({
        'distance': d, 'noise_strengths': NOISE_STRENGTHS,
        'pauli_logical_z': pauli_z, 'coherent_logical_z': coherent_z,
    })

print('\n✅ Noise sweep complete.')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors_p = ['#4CAF50', '#2196F3']
colors_c = ['#FF5722', '#E91E63']
markers = ['o', 's']

for i, r in enumerate(all_results):
    d = r['distance']
    ax.plot(r['noise_strengths'], r['pauli_logical_z'],
            f'{markers[i]}--', color=colors_p[i], linewidth=2,
            markersize=8, alpha=0.85, label=f'Pauli (d={d})')
    ax.plot(r['noise_strengths'], r['coherent_logical_z'],
            f'{markers[i]}-', color=colors_c[i], linewidth=2.5,
            markersize=8, label=f'Coherent (d={d})')

ax.axhline(y=1.0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Noise Strength (ε = p)', fontsize=13)
ax.set_ylabel('Logical Z Fidelity ⟨Z_L⟩', fontsize=13)
ax.set_title('Coherent vs Pauli Noise on Surface Code Topology\n'
             'Coherent errors accumulate — invisible to Stim',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='lower left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('qec_noise_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 2. Multi-Backend Comparison

Run the **same** Pauli noise circuit on Stabilizer, PauliPropagator, and MPS — then add coherent noise (MPS only).

In [ ]:
d = 5
model = SurfaceCodeModel(d)
data_obs = build_data_only_observables(model.n_data)
logical_z_obs = build_logical_z(model)
all_obs = data_obs + [logical_z_obs]

p = 0.03
n_avg = 10

backends = [
    ('PauliPropagator', maestro.SimulationType.PauliPropagator),
    ('Stabilizer', maestro.SimulationType.Stabilizer),
    ('MPS (Pauli)', maestro.SimulationType.MatrixProductState),
]

results = {}
print(f'Backend comparison: d={d}, p={p}, {n_avg} samples\n')

for name, sim_type in backends:
    all_logical = []
    t0 = time.perf_counter()
    for seed in range(n_avg):
        qc = model.build_clifford_cx_network(
            n_rounds=N_ROUNDS, noise_strength=p, seed=seed)
        config_kwargs = {'simulator_type': maestro.SimulatorType.QCSim,
                         'simulation_type': sim_type}
        if sim_type == maestro.SimulationType.MatrixProductState:
            config_kwargs['max_bond_dimension'] = CHI
        result = qc.estimate(all_obs, maestro.SimulatorConfig(**config_kwargs))
        all_logical.append(result['expectation_values'][-1])
    elapsed_ms = (time.perf_counter() - t0) * 1000
    results[name] = {'logical_z': float(np.mean(all_logical)), 'time_ms': elapsed_ms}
    print(f'  {name:<22} ⟨Z_L⟩={results[name]["logical_z"]:+.6f}  {elapsed_ms:.1f}ms')

# Coherent noise — only MPS can do this
t0 = time.perf_counter()
qc_coh = model.build_noisy_cx_network(
    n_rounds=N_ROUNDS, noise_type='coherent', noise_strength=p)
result_coh = qc_coh.estimate(all_obs, maestro.SimulatorConfig(
    simulator_type=SIM_TYPE,
    simulation_type=maestro.SimulationType.MatrixProductState,
    max_bond_dimension=CHI))
elapsed_ms = (time.perf_counter() - t0) * 1000
results['MPS (Coherent)'] = {
    'logical_z': result_coh['expectation_values'][-1],
    'time_ms': elapsed_ms}
print(f'  {"MPS (Coherent) ★":<22} ⟨Z_L⟩={results["MPS (Coherent)"]["logical_z"]:+.6f}  {elapsed_ms:.1f}ms')
print(f'\n★ Only MPS can simulate coherent noise.')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

names = list(results.keys())
logical_vals = [results[n]['logical_z'] for n in names]
times = [results[n]['time_ms'] for n in names]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#FF5722']
x = np.arange(len(names))

ax1.bar(x, logical_vals, color=colors[:len(names)], alpha=0.85)
ax1.set_xticks(x)
ax1.set_xticklabels(names, rotation=15, ha='right', fontsize=10)
ax1.set_ylabel('Logical Z ⟨Z_L⟩', fontsize=12)
ax1.set_title(f'Backend Comparison (d={d}, p={p})', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
for i, v in enumerate(logical_vals):
    ax1.text(i, v + 0.01, f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')

ax2.bar(x, times, color=colors[:len(names)], alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(names, rotation=15, ha='right', fontsize=10)
ax2.set_ylabel('Time (ms)', fontsize=12)
ax2.set_title('Simulation Speed', fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
for i, t in enumerate(times):
    ax2.text(i, t + max(times)*0.02, f'{t:.1f}ms', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('qec_backend_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 3. Spatial Error Structure

Heatmaps showing per-data-qubit error deviation. Coherent noise creates **structured, spatially correlated** patterns; Pauli noise is uniform.

In [ ]:
d = 5
model = SurfaceCodeModel(d)
p = 0.03

pauli = measure_noise_impact(model, 'pauli', p, n_samples=N_SAMPLES)
coherent = measure_noise_impact(model, 'coherent', p, n_samples=1)

pauli_dev = np.abs(1.0 - pauli['data_z']).reshape(d, d)
coherent_dev = np.abs(1.0 - coherent['data_z']).reshape(d, d)

vmax = max(np.max(pauli_dev), np.max(coherent_dev)) * 1.1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

im1 = ax1.imshow(pauli_dev, cmap='YlOrRd', vmin=0, vmax=vmax, aspect='equal')
ax1.set_title(f'Pauli Noise (ε={p})\nRandom, uniform', fontsize=12, fontweight='bold')
plt.colorbar(im1, ax=ax1, label='Error deviation (1 − ⟨Z⟩)', shrink=0.8)
for r in range(d):
    for c in range(d):
        ax1.text(c, r, f'{pauli_dev[r,c]:.3f}', ha='center', va='center', fontsize=8,
                 color='white' if pauli_dev[r,c] > vmax*0.6 else 'black')

im2 = ax2.imshow(coherent_dev, cmap='YlOrRd', vmin=0, vmax=vmax, aspect='equal')
ax2.set_title(f'Coherent Noise (ε={p})\nStructured, correlated', fontsize=12, fontweight='bold')
plt.colorbar(im2, ax=ax2, label='Error deviation (1 − ⟨Z⟩)', shrink=0.8)
for r in range(d):
    for c in range(d):
        ax2.text(c, r, f'{coherent_dev[r,c]:.3f}', ha='center', va='center', fontsize=8,
                 color='white' if coherent_dev[r,c] > vmax*0.6 else 'black')

fig.suptitle(f'Surface Code d={d}: Data Qubit Error Structure', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('qec_syndrome_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---

## GPU Mode

Set `USE_GPU = True` in the first cell and increase `CHI` for higher-fidelity simulation at larger distances.

MPS tensor contractions scale as **O(χ³)** — GPU acceleration is critical for d=7+ (97+ qubits).

👉 **[Start your free GPU trial at maestro.qoroquantum.net](https://maestro.qoroquantum.net)**